# 08 — Constrained multi-objective Bayesian optimisation

This notebook adds a controlled formulation/application search problem. The optimiser must balance
**efficacy** and **environmental burden** while respecting a hard **crop-injury** feasibility limit.

The response surfaces are synthetic and known only to the evaluation harness. They are not product
recommendations and do not represent any real formulation.

In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from crop_protection_ps.multiobjective_demo import run_multiobjective_demo

summary = run_multiobjective_demo(ROOT)
summary["scope"]

{'status': 'controlled synthetic formulation/application optimisation experiment',
 'grid_candidates': 1800,
 'initial_evaluations': 12,
 'sequential_evaluations': 18,
 'total_evaluations_per_policy': 30,
 'n_rollouts': 50,
 'injury_limit': 0.34,
 'warning': 'Response surfaces, thresholds and design variables are controlled simulation parameters, not claims about a real formulation or Crop Protection product.'}

## Decision structure

For condition \(x\), the controlled endpoints are

\[
E(x)=	ext{efficacy},\qquad
B(x)=	ext{environmental burden},\qquad
I(x)=	ext{crop injury}.
\]

The search problem is

\[
\max E(x),\qquad \min B(x),\qquad I(x)\le 0.34.
\]

The multi-objective policy uses scalarised expected improvement multiplied by posterior feasibility
probability. All policies receive exactly the same number of experiments.

In [2]:
metrics_path = ROOT / "results" / "multiobjective_bo" / "policy_metrics.csv"
metrics: pd.DataFrame = pd.read_csv(metrics_path)
metrics[[
    "policy",
    "mean_hypervolume_ratio",
    "mean_unsafe_evaluation_rate",
    "mean_oracle_frontier_recall",
    "mean_n_balanced_high_value_evaluated",
]]

,policy,mean_hypervolume_ratio,mean_unsafe_evaluation_rate,mean_oracle_frontier_recall,mean_n_balanced_high_value_evaluated
0,random,0.834813,0.250667,0.012683,0.38
1,efficacy_only,0.831720,0.296000,0.039512,0.56
2,constrained_pareto,0.901633,0.194000,0.058049,1.28


The primary metric is the fraction of the hidden **feasible Pareto hypervolume** recovered by the
conditions actually evaluated. This avoids collapsing efficacy and environmental burden into one
fixed utility after the optimisation is complete.

In [3]:
gate_path = ROOT / "results" / "multiobjective_bo" / "promotion_gate.json"
import json

promotion_gate: dict[str, object] = json.loads(gate_path.read_text(encoding="utf-8"))
promotion_gate

{'minimum_hypervolume_improvement_vs_efficacy_only_percent': 8.0,
 'observed_hypervolume_improvement_vs_efficacy_only_percent': 8.405884784479223,
 'paired_hypervolume_ratio_difference_constrained_pareto_minus_efficacy_only': {'mean': 0.069913412357347,
  'mc95_low': 0.06352686932603514,
  'mc95_high': 0.07629995538865884},
 'paired_unsafe_rate_difference_constrained_pareto_minus_random': {'mean': -0.05666666666666667,
  'mc95_low': -0.07788845840168387,
  'mc95_high': -0.035444874931649474},
 'promoted': True}

## Interpretation

The locked experiment promotes the constrained multi-objective policy only if it improves mean
hypervolume by at least 8% versus efficacy-only Bayesian search **and** the paired Monte Carlo 95%
interval for that difference remains above zero.

The important distinction is

\[
oxed{	ext{best potency}
eq	ext{best feasible scientific trade-off}}.
\]

A potent condition can still be unattractive when environmental burden is poor or crop injury is
likely to violate the feasibility limit.

In [4]:
frontier_path = ROOT / "results" / "multiobjective_bo" / "oracle_feasible_pareto_frontier.csv"
frontier: pd.DataFrame = pd.read_csv(frontier_path)
frontier.sort_values("efficacy", ascending=False).head(10)

,candidate,dose,formulation,adjuvant,efficacy,environmental_burden,crop_injury
37,1035,0.592857,0.636364,0.555556,1.000000,0.548695,0.229144
36,1034,0.592857,0.636364,0.444444,0.996759,0.537584,0.183545
35,1025,0.592857,0.545455,0.555556,0.986681,0.525220,0.218093
31,915,0.525000,0.636364,0.555556,0.981015,0.506315,0.204717
30,914,0.525000,0.636364,0.444444,0.940537,0.495204,0.160626
29,905,0.525000,0.545455,0.555556,0.930195,0.483272,0.193666
25,795,0.457143,0.636364,0.555556,0.906312,0.463935,0.181753
40,1225,0.728571,0.181818,0.555556,0.891146,0.463361,0.291004
39,1115,0.660714,0.272727,0.555556,0.888983,0.458054,0.255638
24,794,0.457143,0.636364,0.444444,0.866099,0.452824,0.139171


## Boundary of the demonstration

The optimiser deliberately uses independent Bayesian response surfaces and a ParEGO-style
scalarisation rather than claiming exact expected hypervolume improvement. The finite hidden grid,
true Pareto frontier, and synthetic response surfaces exist only so the decision policy can be
scored against known truth.

A production Crop Protection optimisation would need domain-specific endpoints, correlated response
models, physically meaningful formulation variables, safety/regulatory constraints and expert review.